# dcgan-wrapper-netG-netD — faded example 2: Complete the discriminator's optimizer

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dcgan-wrapper-netG-netD`. The last cell reports your progress on the `Generative: DCGAN netG+netD wrapper` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Generative: DCGAN netG+netD wrapper` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dcgan-wrapper-netG-netD`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dcgan-wrapper-netG-netD"
DD_SUBTOPIC = "Generative: DCGAN netG+netD wrapper"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

With `netG` and `netD` as separate submodules of the wrapper, GAN training builds one optimizer per subnet. Each optimizer is given only that subnet's parameter iterator (`wrapper.netD.parameters()`), keeping the two parameter sets disjoint so a discriminator step never touches generator weights.

## Faded exercise 2

### Faded — give the discriminator its own optimizer

Complete `make_optimizers(generator, discriminator)`. The generator's optimizer is already built from `wrapper.netG.parameters()`. Build the discriminator's optimizer the same way — an `Adam` over only the discriminator subnet's parameters, with `lr=2e-4` and `betas=(0.5, 0.999)`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
from torch import nn
import torch as t

def make_optimizers(generator, discriminator):
    class DCGAN(nn.Module):
        def __init__(self, netG, netD):
            super().__init__()
            self.netG = netG
            self.netD = netD
    wrapper = DCGAN(generator, discriminator)
    optG = t.optim.Adam(wrapper.netG.parameters(), lr=2e-4, betas=(0.5, 0.999))
    optD = None  # TODO: fill in this step — read the prompt cell above
    return wrapper, optG, optD

t.manual_seed(0)
gen = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 64))
disc = nn.Sequential(nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1))
wrapper, optG, optD = make_optimizers(gen, disc)


def _test():
    from torch import nn
    import torch as t
    t.manual_seed(0)
    g = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 64))
    d = nn.Sequential(nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1))
    w, oG, oD = make_optimizers(g, d)
    assert isinstance(oD, t.optim.Adam)
    g_ids = {id(p) for grp in oG.param_groups for p in grp['params']}
    d_ids = {id(p) for grp in oD.param_groups for p in grp['params']}
    assert g_ids.isdisjoint(d_ids), 'optimizers must not share parameters'
    d_expected = {id(p) for p in w.netD.parameters()}
    assert d_ids == d_expected, 'optD must cover exactly the discriminator params'
    grp = oD.param_groups[0]
    assert abs(grp['lr'] - 2e-4) < 1e-12
    assert grp['betas'] == (0.5, 0.999)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from torch import nn
import torch as t

def make_optimizers(generator, discriminator):
    class DCGAN(nn.Module):
        def __init__(self, netG, netD):
            super().__init__()
            self.netG = netG
            self.netD = netD
    wrapper = DCGAN(generator, discriminator)
    optG = t.optim.Adam(wrapper.netG.parameters(), lr=2e-4, betas=(0.5, 0.999))
    optD = t.optim.Adam(wrapper.netD.parameters(), lr=2e-4, betas=(0.5, 0.999))
    return wrapper, optG, optD

t.manual_seed(0)
gen = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 64))
disc = nn.Sequential(nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1))
wrapper, optG, optD = make_optimizers(gen, disc)
```
</details>